Notebook to read/compare the output parquet files from the `output-transit-eet` and `output-base-eet` runs we have.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
base_folder = Path("/Users/tomstephen/dev/asim_eet_viz/output-base-eet")
transit_folder = Path("/Users/tomstephen/dev/asim_eet_viz/output-transit-eet")

poi = 1191 # person of interest 1191 is the og (adult), 1192 & 1193 & 1194 are children (ages 8, 1, 1)
hhoi = 467 # household of interest

In [68]:
# other households we may be interested in: 5308, 6008
# need to find the person ids for those households, then we can compare their tours as well
def find_persons_in_household(folder, household_id):
    persons = pd.read_parquet(folder / "final_persons.parquet")
    return persons[persons["household_id"] == household_id]

# Example usage:
hh_id = 6008
base_hh_persons = find_persons_in_household(base_folder, hh_id)
transit_hh_persons = find_persons_in_household(transit_folder, hh_id)

display(f"Household {hh_id} — person_ids:", base_hh_persons.index.tolist())

'Household 6008 — person_ids:'

[15618, 15619, 15620, 15621, 15622, 15623]

In [69]:
def read_persons(folder):
    return pd.read_parquet(folder / "final_persons.parquet")

base_persons = read_persons(base_folder)
transit_persons = read_persons(transit_folder)

# poi person_id only (person_id is the index)
base_poi = base_persons.loc[[poi]]
transit_poi = transit_persons.loc[[poi]]

# append "scenario" column
base_poi["scenario"] = "base"
transit_poi["scenario"] = "transit"

# join them together for easier comparison
merged = pd.concat([base_poi, transit_poi], ignore_index=True)

# compute the columns where they differ
diff_mask = merged[merged.duplicated(subset=merged.columns.difference(["scenario"]), keep=False) == False].index

# and just show different columns
diff_columns = merged.columns[merged.loc[diff_mask].nunique() > 1]

# show it
display(base_poi[diff_columns])
display(transit_poi[diff_columns])

""
person_id
15623


""
person_id
15623


In [70]:
def read_households(folder):
    return pd.read_parquet(folder / "final_households.parquet")

base_households = read_households(base_folder)
transit_households = read_households(transit_folder)

# just hoi, then join, then compute differences
base_hoi = base_households.loc[[hhoi]]
transit_hoi = transit_households.loc[[hhoi]]
# append "scenario" column
base_hoi["scenario"] = "base"
transit_hoi["scenario"] = "transit"
merged = pd.concat([base_hoi, transit_hoi], ignore_index=True)
diff_mask = merged[merged.duplicated(subset=merged.columns.difference(["scenario"]), keep=False) == False].index
diff_columns = merged.columns[merged.loc[diff_mask].nunique() > 1]
display(base_hoi[diff_columns])
display(transit_hoi[diff_columns])

,school_escorting_outbound,school_escorting_outbound_cond,scenario
household_id,,,
6008,8.0,24.0,base


,school_escorting_outbound,school_escorting_outbound_cond,scenario
household_id,,,
6008,1.0,1.0,transit


In [71]:
def read_tours(folder):
    return pd.read_parquet(folder / "final_tours.parquet")

base_tours = read_tours(base_folder)
transit_tours = read_tours(transit_folder)

# tours for person of interest
base_poi_tours = base_tours[base_tours["person_id"] == poi]
transit_poi_tours = transit_tours[transit_tours["person_id"] == poi]

print(f"Person {poi} tours: base has {len(base_poi_tours)}, transit has {len(transit_poi_tours)}")
display(base_poi_tours)
display(transit_poi_tours)


Person 15623 tours: base has 1, transit has 1


,person_id,tour_type,tour_type_count,tour_type_num,tour_num,tour_count,tour_category,number_of_participants,destination,origin,...,vehicle_occup_1,vehicle_occup_2,vehicle_occup_3.5,tour_mode,mode_choice_logsum,selected_vehicle,atwork_subtour_frequency,parent_tour_id,stop_frequency,primary_purpose
tour_id,,,,,,,,,,,,,,,,,,,,,
781183,15623,school,1,1,1,1,mandatory,1,5029.0,5702.0,...,Van_9_Gas,SUV_13_Gas,Van_9_Gas,SHARED3,-9.261444,Van_9_Gas,,NaN,1out_0in,school


,person_id,tour_type,tour_type_count,tour_type_num,tour_num,tour_count,tour_category,number_of_participants,destination,origin,...,vehicle_occup_1,vehicle_occup_2,vehicle_occup_3.5,tour_mode,mode_choice_logsum,selected_vehicle,atwork_subtour_frequency,parent_tour_id,stop_frequency,primary_purpose
tour_id,,,,,,,,,,,,,,,,,,,,,
781183,15623,school,1,1,1,1,mandatory,1,5029.0,5702.0,...,Van_9_Gas,SUV_13_Gas,Van_9_Gas,SHARED3,-9.261478,Van_9_Gas,,NaN,1out_0in,school


In [72]:
def read_proto_disaggregate_accessibility(folder):
    return pd.read_parquet(folder / "final_proto_disaggregate_accessibility.parquet")

base_pdacc = read_proto_disaggregate_accessibility(base_folder)
transit_pdacc = read_proto_disaggregate_accessibility(transit_folder)

# filter by proto_person_id (which matches person_id in final_persons)
# its also the index
base_pdacc_poi = base_pdacc.loc[[poi]]
transit_pdacc_poi = transit_pdacc.loc[[poi]]

# get columns with differences
pdacc_cols = [c for c in base_pdacc.columns if c in transit_pdacc.columns]
merged = pd.concat([base_pdacc_poi[pdacc_cols].T, transit_pdacc_poi[pdacc_cols].T], axis=1, keys=["base", "transit"])
merged["diff"] = ~(
    (merged["base"] == merged["transit"])
    | (merged["base"].isna() & merged["transit"].isna())
)
diff_cols = merged[merged["diff"]].index.tolist()
print(f"Person {poi} proto disaggregate accessibility differences: {len(diff_cols)} columns differ")
if len(diff_cols) > 0:
    display(merged.loc[diff_cols])

Person 15623 proto disaggregate accessibility differences: 6 columns differ


,base,transit,diff
proto_person_id,15623,15623,
workplace_location_accessibility,13.082305,13.083396,True
othdiscr_accessibility,14.045611,14.057429,True
shopping_accessibility,10.452464,10.455865,True
workplace_location_accessibility_1,13.082305,13.083396,True
othdiscr_accessibility_1,14.045611,14.057429,True
shopping_accessibility_1,10.452464,10.455865,True
